In [16]:
import numpy as np 
import pandas as pd 
import matplotlib. pyplot as plt
import seaborn as sns

In [17]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])

In [18]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [19]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [20]:
unique_emotion = df["emotion"].unique()  #coverting the emotion into the numeric values
emotion_numbers ={}
i =  0
for emo in unique_emotion:
    emotion_numbers [emo]=i
    i +=1
df['emotion'] = df['emotion'].map (emotion_numbers)
df  

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [21]:
 df["text"]= df["text"].apply(lambda x : x.lower())

In [22]:
import string 
def remove_punc(txt) :
    return txt. translate(str.maketrans('','', string.punctuation))

In [23]:
df["text"]=df["text"].apply(remove_punc)

In [24]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [25]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [26]:
import nltk 

In [27]:
from nltk. corpus import stopwords
from nltk. tokenize import word_tokenize

In [28]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/patelsoham/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/patelsoham/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
stop_words = set(stopwords.words('english'))
stop_words.head()

AttributeError: 'set' object has no attribute 'head'

In [30]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [31]:
df['text'] = df['text'].apply(remove)

In [32]:

df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [33]:
#Bag of Words (BoW) encoding
from sklearn.feature_extraction. text import CountVectorizer
documents= [
"I lovepizza"
"Pizza is the best",
"I love pasta",
"Pasta is great"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
print("Vocabulary:", vectorizer.get_feature_names_out () )
print ("\nBoW Matrix: \n", X.toarray())

Vocabulary: ['best' 'great' 'is' 'love' 'lovepizzapizza' 'pasta' 'the']

BoW Matrix: 
 [[1 0 1 0 1 0 1]
 [0 0 0 1 0 1 0]
 [0 1 1 0 0 1 0]]


In [34]:
from sklearn.feature_extraction. text import CountVectorizer
documents= [
"I lovepizza"
"Pizza is the best",
"I love pasta",
"Pasta is great"]
vectorizer = CountVectorizer(ngram_range=(3,3))
X = vectorizer.fit_transform(documents)
print("Vocabulary:", vectorizer.get_feature_names_out () )
print ("\nBoW Matrix: \n", X.toarray())

Vocabulary: ['is the best' 'lovepizzapizza is the' 'pasta is great']

BoW Matrix: 
 [[1 1 0]
 [0 0 0]
 [0 0 1]]


In [35]:
#Term frequency(TF) * Inverse document frequency (IDF) Encoding 
from sklearn.feature_extraction. text import TfidfVectorizer
documents= [
"I lovepizza"
"Pizza is the best",
"I love pasta",
"Pasta is great"]
vectorizer =  TfidfVectorizer(ngram_range=(2,2))
X = vectorizer.fit_transform(documents)
print("Vocabulary:", vectorizer.get_feature_names_out () )
print ("\nBoW Matrix: \n", X.toarray())

Vocabulary: ['is great' 'is the' 'love pasta' 'lovepizzapizza is' 'pasta is'
 'the best']

BoW Matrix: 
 [[0.         0.57735027 0.         0.57735027 0.         0.57735027]
 [0.         0.         1.         0.         0.         0.        ]
 [0.70710678 0.         0.         0.         0.70710678 0.        ]]


In [36]:
#contnuing projects
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)
     

In [37]:
#Naive Bayes (NB)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


In [38]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [40]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)
y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.6609375


In [41]:
#By LogisticRegression
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf,y_train)
log_pred = logistic_model.predict(X_test_tfidf)
print(accuracy_score(y_test,log_pred ))

0.8628125
